# RLHF Data Generation with Trained LoRA
This notebook systematically generates image sets for Reinforcement Learning from Human Feedback (RLHF).

**Key steps:**
1.  Load the base Stable Diffusion model and our trained LoRA adapter.
2.  **Load Prompts from external file (`prompts_age_inc_5.txt`).**
3.  For each prompt, generate **two image samples (A and B)** for pairwise ranking.
4.  For each sample, save the final image, the **initial latent tensor** for DPO, and collect all generation metadata.
5.  **Save all metadata to a single `generation_manifest.jsonl` file.**

**Key features for Scalability and Resilience:**
1.  **Distributed Generation:** Define a range (`START_INDEX`, `END_INDEX`) to process a subset of the full prompt list.
2.  **Resilience:** Metadata is **appended** to the `generation_manifest.jsonl` file after *each* sample is successfully generated.
3.  **Conflict Resolution:** Existing samples found in the manifest are **skipped**, allowing runs to be easily resumed or split across machines.
 
---

In [ ]:
# %%
# Essential Imports
import os 
import gc
import json
import hashlib
from pathlib import Path
from typing import List, Dict, Optional, Any, Set
from PIL import Image as PILImage
import torch
from diffusers import StableDiffusionPipeline
from tqdm.auto import tqdm

In [ ]:
# %%
# --- Setup and Path Configuration (Relative to project_path) ---

# The directory where you want to save the generated data for RLHF
# Assuming 'ratings_interface' is a folder in your project root
RLHF_DATA_DIR = Path("./ratings_interface/rlhf_generation_data")
RLHF_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Path to your list of structured prompts
PROMPTS_FILE = Path("./prompts_age_inc_5.txt") 

# --- Distributed Generation Range Settings (CRITICAL FOR SPLITTING WORK) ---
# Set the range of prompts (by index) to process in this run.
# e.g., Job 1: START_INDEX = 0, END_INDEX = 1000
# e.g., Job 2: START_INDEX = 1000, END_INDEX = None
START_INDEX = 0 
END_INDEX = None 

# Path to your trained LoRA model
LORA_CHECKPOINT_BASE_DIR = Path("./lora_training_runs")
RESUME_RUN_ID = '519037_run_20250903-160027' 
LORA_PATH = LORA_CHECKPOINT_BASE_DIR / RESUME_RUN_ID / "lora_checkpoints/best_lora_adapter"
LORA_WEIGHT_NAME = "adapter_model.safetensors" # Required fix for safetensors loading

In [ ]:
# %%
# --------------------------------------------------------------------------------------------------
## Core Generation Functions (Refactored for RLHF)
# --------------------------------------------------------------------------------------------------

def load_prompts_from_file(file_path: Path) -> List[Dict[str, str]]:
    """Loads prompts from a file and pre-calculates the deterministic hash for each."""
    if not file_path.exists():
        print(f"❌ Error: Prompt file not found at {file_path}. Returning empty list.")
        return []
    
    with open(file_path, 'r') as f:
        prompts = [line.strip() for line in f if line.strip()]
    
    # Pre-calculate hash and combine with prompt text
    prompt_list_with_hash = []
    for prompt in prompts:
        prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()[:10]
        prompt_list_with_hash.append({
            "prompt": prompt,
            "prompt_hash": prompt_hash
        })
    
    print(f"✅ Loaded {len(prompt_list_with_hash)} total prompts from {file_path.name}.")
    return prompt_list_with_hash


def load_lora_for_inference(
    lora_adapter_path: Path,
    weight_name: str,
    device: str = 'cuda'
) -> Optional[StableDiffusionPipeline]:
    """Loads the Stable Diffusion pipeline and applies the trained LoRA weights."""
    if not lora_adapter_path.is_dir():
        print(f"❌ Error: LoRA adapter path must be a directory. Path not found at {lora_adapter_path}.")
        return None

    print("🚀 Loading base Stable Diffusion model...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False
    )

    print(f"✅ Applying LoRA weights from {lora_adapter_path} using file {weight_name}")
    # CRITICAL FIX: Pass the weight_name argument to load the safetensors file
    pipe.unet.load_attn_procs(lora_adapter_path, weight_name=weight_name) 
    pipe = pipe.to(device)
    return pipe


def load_existing_samples(manifest_path: Path) -> Set[str]:
    """Loads all existing sample_ids from the manifest to prevent re-generation."""
    existing_samples = set()
    if manifest_path.exists():
        with open(manifest_path, 'r') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    # Check for sample_id which is the unique identifier (e.g., 'd2a9f12b3c_A')
                    if 'sample_id' in data:
                        existing_samples.add(data['sample_id'])
                except json.JSONDecodeError:
                    continue 
    return existing_samples


def generate_and_save_samples(
    pipe: StableDiffusionPipeline,
    prompt: str,
    prompt_hash: str,
    output_dir: Path,
    existing_samples: Set[str], # Passed for conflict checking
    manifest_file, # Passed for immediate appending
    base_output_dir: Path, # The top-level RLHF_DATA_DIR
    num_samples: int = 2,
    num_inference_steps: int = 20,
    guidance_scale: float = 7.5,
):
    """
    Generates samples for a single prompt, skips existing ones, and saves artifacts immediately.
    """
    pipe.unet.eval()
    
    # Get dimensions from the model config
    height = pipe.unet.config.sample_size * pipe.vae_scale_factor
    width = pipe.unet.config.sample_size * pipe.vae_scale_factor
    device = pipe.device
    
    for i in range(num_samples):
        sample_letter = chr(ord('A') + i)
        sample_base_name = f"{prompt_hash}_{sample_letter}"
        
        # --- Conflict Check (Skip if already generated) ---
        if sample_base_name in existing_samples:
            continue 
        
        # --- Generation Setup ---
        seed = torch.randint(0, 2**32 - 1, (1,)).item()
        generator = torch.Generator(device=device).manual_seed(seed)

        # Create the initial latent noise
        latents = torch.randn(
            (1, pipe.unet.config.in_channels, int(height // 8), int(width // 8)),
            generator=generator,
            device=device,
            dtype=torch.float16
        )

        # --- Generate Image ---
        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type):
                result = pipe(
                    prompt,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale,
                    generator=generator,
                    latents=latents,
                )
        image = result.images[0]

        # --- Save Artifacts ---
        image_path = output_dir / f"{sample_base_name}.png"
        latent_path = output_dir / f"{sample_base_name}_latent.pt"

        image.save(image_path)
        # We move it to cpu to save space and avoid GPU memory issues
        torch.save(latents.cpu(), latent_path) 

        # --- Save Metadata (Immediate Append) ---
        metadata = {
            "prompt_hash": prompt_hash,
            "prompt": prompt,
            "seed": seed,
            "sample_id": sample_base_name,
            "num_inference_steps": num_inference_steps,
            "guidance_scale": guidance_scale,
            # Use relative paths for portability, relative to the base_output_dir
            "image_path": os.path.relpath(image_path, base_output_dir), 
            "latent_path": os.path.relpath(latent_path, base_output_dir), 
        }
        
        # Write immediately to manifest (resilience)
        manifest_file.write(json.dumps(metadata) + '\n')
        manifest_file.flush() # Ensure it's written to disk immediately
        
        # Update set to prevent re-generation later in this run
        existing_samples.add(sample_base_name)


def generate_rlhf_dataset(
    pipe: StableDiffusionPipeline,
    prompts: List[Dict[str, str]],
    base_output_dir: Path,
    start_index: int,
    end_index: Optional[int],
    num_samples_per_prompt: int = 2,
    ):
    """Orchestrates the generation of the RLHF dataset, applying range filtering and saving incrementally."""
    
    # 1. Apply range filtering based on user settings
    prompts_to_run = prompts[start_index:end_index]
    
    print(f"✨ Starting RLHF data generation...")
    print(f"Processing range: Index {start_index} to {start_index + len(prompts_to_run) - 1}")
    
    # 2. Load existing samples for conflict resolution
    manifest_path = base_output_dir / "generation_manifest.jsonl"
    existing_samples = load_existing_samples(manifest_path)
    print(f"Found {len(existing_samples)} samples already in manifest. They will be skipped.")
    
    # 3. Open manifest file in append mode for incremental saving
    with open(manifest_path, 'a') as manifest_file:
        
        for prompt_data in tqdm(prompts_to_run, desc=f"Generating prompts"):
            
            prompt = prompt_data['prompt']
            prompt_hash = prompt_data['prompt_hash']
            prompt_dir = base_output_dir / prompt_hash
            prompt_dir.mkdir(exist_ok=True)
            
            generate_and_save_samples(
                pipe=pipe,
                prompt=prompt,
                prompt_hash=prompt_hash,
                output_dir=prompt_dir,
                existing_samples=existing_samples,
                manifest_file=manifest_file, # Pass the file handler
                base_output_dir=base_output_dir, # Pass for correct relative path calculation
                num_samples=num_samples_per_prompt,
            )
        
    print(f"\n✅ Generation session complete! Data saved in: {base_output_dir}")


def unload_model(pipe: StableDiffusionPipeline) -> None:
    """Unloads the pipeline and clears GPU memory."""
    print("🔻 Unloading pipeline to free GPU memory...")
    del pipe
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ Model unloaded and GPU cache cleared.")


In [ ]:
# %%
# --------------------------------------------------------------------------------------------------
## Main Execution Block
# --------------------------------------------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load LoRA Weights and Model ----
pipeline = load_lora_for_inference(LORA_PATH, LORA_WEIGHT_NAME, device) 

if pipeline is None:
    print("❌ Failed to load the model. Halting execution.")
else:
    # ---- Define the Prompts for Generation ----
    # Loads all prompts and pre-calculates the deterministic hash for each
    prompts_with_hash = load_prompts_from_file(PROMPTS_FILE)

    if not prompts_with_hash:
        print("🛑 No prompts loaded. Halting execution.")
    else:
        # ---- Run the Dataset Generation ----
        generate_rlhf_dataset(
            pipe=pipeline,
            prompts=prompts_with_hash,
            base_output_dir=RLHF_DATA_DIR,
            start_index=START_INDEX, # Range setting
            end_index=END_INDEX,     # Range setting
            num_samples_per_prompt=2 
        )

        # ---- Unload Model ----
        unload_model(pipeline)